# Hospitals - Silver Layer (dim_hospital)

Ingestion incrémentale depuis Bronze avec déduplication sur `hospital_id`. Utilise `foreachBatch` + `MERGE` pour upsert.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

CATALOG    = "hospital"
SOURCE     = f"{CATALOG}.bronze.hospital_raw"
TARGET     = f"{CATALOG}.silver.dim_hospital"
QUARANTINE = f"{CATALOG}.silver.quarantine_hospital"
CHECKPOINT = f"/Volumes/{CATALOG}/bronze/checkpoints/silver/dim_hospital"

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TARGET} (
  hospital_id    STRING NOT NULL,
  hospital_name  STRING,
  city           STRING,
  bed_count      INT,
  row_hash       STRING,
  valid_from     TIMESTAMP,
  valid_to       TIMESTAMP,
  is_current     BOOLEAN,
  _source_file   STRING,
  _updated_at    TIMESTAMP
) COMMENT 'Dimension hôpital (SCD2)'
""")

In [0]:
def upsert_dim_hospital(batch_df, batch_id):
    # 1. Typage et nettoyage
    typed = batch_df.select(
        F.upper(F.trim("hospital_id")).alias("hospital_id"),
        F.trim("hospital_name").alias("hospital_name"),
        F.trim("city").alias("city"),
        F.expr("try_cast(bed_count AS INT)").alias("bed_count"),
        "_rescued_data", "_source_file", "_ingested_at")

    # 2. Qualité
    reason = (F.when(F.col("hospital_id").isNull() | (F.col("hospital_id") == ""), "hospital_id manquant")
               .when(F.col("bed_count").isNull() | (F.col("bed_count") < 0), "bed_count invalide")
               .when(F.col("_rescued_data").isNotNull(), "données inattendues"))
    checked = typed.withColumn("_reject_reason", reason)

    (checked.filter("_reject_reason IS NOT NULL")
        .withColumn("_rejected_at", F.current_timestamp())
        .write.mode("append")
        .option("txnAppId", "quarantine_hospital").option("txnVersion", batch_id)
        .saveAsTable(QUARANTINE))

    # 3. Dernière version par hôpital dans le batch + empreinte des attributs suivis
    w = Window.partitionBy("hospital_id").orderBy(F.col("_ingested_at").desc())
    latest = (checked.filter("_reject_reason IS NULL")
        .withColumn("_rn", F.row_number().over(w)).filter("_rn = 1")
        .withColumn("row_hash", F.sha2(F.concat_ws("||", "hospital_name", "city",
                                                    F.col("bed_count").cast("string")), 256)))

    # 4. Comparer avec la version courante
    current = (spark.table(TARGET).filter("is_current")
               .select("hospital_id", F.col("row_hash").alias("current_hash")))
    joined = latest.join(current, "hospital_id", "left")

    new_rows = (joined.filter("current_hash IS NULL")                        # hôpital inconnu
                .withColumn("valid_from", F.lit("1900-01-01").cast("timestamp")))
    changed  = (joined.filter("current_hash IS NOT NULL AND current_hash <> row_hash")  # attribut modifié
                .withColumn("valid_from", F.col("_ingested_at")))

    # 5. Préparer le MERGE : une ligne pour FERMER l'ancienne version, une pour INSÉRER la nouvelle
    cols = ["hospital_id", "hospital_name", "city", "bed_count", "row_hash", "valid_from", "_source_file"]
    staged = (new_rows.select(F.col("hospital_id").alias("merge_key"), *cols)
        .unionByName(changed.select(F.col("hospital_id").alias("merge_key"), *cols))          # → fermeture
        .unionByName(changed.select(F.lit(None).cast("string").alias("merge_key"), *cols)))   # → insertion

    # 6. MERGE SCD2
    (DeltaTable.forName(spark, TARGET).alias("t")
        .merge(staged.alias("s"), "t.hospital_id = s.merge_key AND t.is_current = true")
        .whenMatchedUpdate(
            condition="t.row_hash <> s.row_hash",
            set={"is_current": "false",
                 "valid_to": "s.valid_from",
                 "_updated_at": "current_timestamp()"})
        .whenNotMatchedInsert(values={
            "hospital_id": "s.hospital_id", "hospital_name": "s.hospital_name",
            "city": "s.city", "bed_count": "s.bed_count", "row_hash": "s.row_hash",
            "valid_from": "s.valid_from", "valid_to": "CAST(NULL AS TIMESTAMP)",
            "is_current": "true", "_source_file": "s._source_file",
            "_updated_at": "current_timestamp()"})
        .execute())

In [0]:
(spark.readStream.table(SOURCE)
    .writeStream.foreachBatch(upsert_dim_hospital)
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .start().awaitTermination())

display(spark.table(TARGET).orderBy("hospital_id", "valid_from"))